In [ ]:
!git clone https://github.com/rifatozkurt/FullWaveformInversion

In [ ]:
!git clone -b improve_transformer https://github.com/rifatozkurt/FullWaveformInversion

In [ ]:
# add google drive to colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd FullWaveformInversion

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GiB",
    )

# Generation of pretraining data

In [ ]:
!python scripts/generate_train_data_colab.py \
    --config configs/experimental.yaml \
    --output-dir /content/extended \
    --start-case-id 1000 \
    --number-of-cases 5000 \
    --case-batch-size 4 \
    --no-overwrite

In [ ]:
!zip -r ../extended.zip ../extended

In [ ]:
from google.colab import files
files.download("../extended.zip")

In [ ]:
print("do not sleep or disconnect")

# Pretraining for 15000 sample models:



In [ ]:
from pathlib import Path

root = Path("/content/extended")

missing = [
    i for i in range(15000)
    if not (root / f"material{i}.h5").exists()
    or not (root / f"gradient{i}.h5").exists()
]

print("Missing cases:", len(missing))
print("First missing IDs:", missing[:20])

In [ ]:
!python scripts/temp_pretraining.py \
    --config /content/FullWaveformInversion/configs/extended.yaml \
    --sample-counts 15000 \
    --available-samples 15000 \
    --models unet,segformer \
    --seed 30 \
    --data-dir /content/extended \
    --output-dir /content/FullWaveformInversion/models/improve_transformer \
    --run-dir /content/FullWaveformInversion/runs/improve_transformer/comparative_pretraining_15k

In [ ]:
!cp -r models/ ../drive/MyDrive/output

In [ ]:
!cp -r runs/ ../drive/MyDrive/output

# Evaluation

In [ ]:
!python scripts/compare_unet_transformer.py \
    --config /content/FullWaveformInversion/configs/extended.yaml \
    --data-dir /content/eval \
    --model-dir /content/FullWaveformInversion/models/improve_transformer \
    --cases 15000,15001,15002,15003,15004,15005 \
    --sample-counts 100,250,500,1000,5000,10000,15000 \
    --models unet,segformer,segformer_highres \
    --epochs 15 \
    --output-dir /content/FullWaveformInversion/runs/compare_unet_transformer

In [ ]:
!cp -r runs/ ../drive/MyDrive/output_eval